In [8]:
# Dependencies and the required libs

import fitz  # standard PyMuPDF
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [10]:
# finding the pdfs from local files

location=Path(r"D:\motor laws\motor laws final")

pdf_files=list(location.rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDFs to process.\n")

Found 4 PDFs to process.



In [11]:
pdf_files[0]

WindowsPath('D:/motor laws/motor laws final/central_motor_vehicles_rule_1989_wrd.pdf')

# PARSING

In [12]:
text_splitter = RecursiveCharacterTextSplitter(separators=[ "\n\nChapter","\n\nSection", "\n\n", "\n", ".", " "], 
                                               chunk_size=500,   # Smaller size for legal facts/laws
                                               chunk_overlap=50, # Minimal overlap
                                               length_function=len
                                                 )

In [13]:
all_docs = []

# 3. Process each PDF step-by-step
for idx, pdf_path in enumerate(pdf_files, start=1):
    # Extract a readable name (e.g., "Motor Vehicles Act 1988")
    doc_title = pdf_path.stem.replace("_", " ").title()
    print(f"[{idx}/{len(pdf_files)}] Parsing: {pdf_path.name}...")

    # Open PDF using standard PyMuPDF
    pdf_document = fitz.open(pdf_path)

    for page_num, page in enumerate(pdf_document, start=1):
        page_text = page.get_text()

        if not page_text.strip():
            continue

        # Split the raw text directly (no markdown intermediate step needed)
        page_chunks = text_splitter.split_text(page_text)

        # Attach file and page metadata
        for chunk_text in page_chunks:
            doc = Document(
                page_content=chunk_text,
                metadata={
                    "source": pdf_path.name,
                    "document_title": doc_title,
                    "page": page_num
                }
            )
            all_docs.append(doc)

print(f"Successfully generated {len(all_docs)} legal chunks.")

[1/4] Parsing: central_motor_vehicles_rule_1989_wrd.pdf...
[2/4] Parsing: Indian_Motor_Vehicles_Act_Violations_Sections.pdf...
[3/4] Parsing: motor_vehicles_act_1988.pdf...
[4/4] Parsing: TN_motor_veh_acts_adminstration.pdf...
Successfully generated 1704 legal chunks.


In [15]:
print(all_docs[1703].page_content)

Tamil Nadu which is a traditional automotive manufacturing hub, is looking to transform 
itself into an Electric Vehicle manufacturing and eco-system hub. The Government aims 
for the good environment which is free from air pollution and noise pollution by using 
electric vehicles.


In [16]:
print(all_docs[100].page_content)

possesses the following qualifications, namely:— 
(a) a minimum educational qualification of a pass in the 10th standard, 
(b) a minimum driving experience of five years in addition to a certificate in a course in motor 
mechanics or any other higher qualification in mechanical engineering from an institution 
established by the Central or a State Government or from an institution recognised by the Board 
of Technical Education of a State Government,


# Embeddings and FAISS

In [17]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\Aswin\AppData\Local\Temp\ipykernel_23416\2935578952.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [18]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Aswin\anaconda3\envs\router_rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5362.69it/s]


In [19]:
from langchain_community.vectorstores import FAISS

print("Generating vector embeddings and saving local FAISS index...")

vectorstore = FAISS.from_documents(all_docs, embeddings) # Vectorize all chunks and store in FAISS database

vectorstore.save_local("laws_faiss_index") # Save index to local folder

print("'laws_faiss_index/' directory created with index.faiss and index.pkl!")

Generating vector embeddings and saving local FAISS index...
'laws_faiss_index/' directory created with index.faiss and index.pkl!
